# Tiền xử lý dữ liệu Abalone

Notebook này triển khai bước `Preprocess Data` dựa trên các kết luận đã rút ra từ notebook EDA. Mục tiêu là tạo ra các phiên bản dữ liệu sạch, nhất quán và sẵn sàng cho bước huấn luyện mô hình.

## 1. Định hướng tiền xử lý rút ra từ EDA

- Biến `Sex` là biến phân loại, cần được mã hóa trước khi huấn luyện mô hình.
- Dữ liệu không có giá trị thiếu, nên không cần điền khuyết ở giai đoạn này.
- Dữ liệu có xuất hiện ngoại lệ và độ lệch ở một số biến khối lượng, nên cần chuẩn bị thêm một phương án tiền xử lý bền vững hơn.
- Các mô hình nhạy với thang đo như `KNeighborsRegressor`, `SVR`, `LinearSVR`, `SGDRegressor` và `MLPRegressor` cần một phiên bản dữ liệu đã scale.
- Việc chia dữ liệu phải tuân thủ yêu cầu đề bài: `70% train`, `30% test`, `random_state = 42`.

## 2. Chuẩn bị thư viện

In [13]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

warnings.filterwarnings('ignore')

## 3. Load dữ liệu gốc

In [14]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'

columns = [
    'Sex',
    'Length',
    'Diameter',
    'Height',
    'WholeWeight',
    'ShuckedWeight',
    'VisceraWeight',
    'ShellWeight',
    'Rings',
]

df = pd.read_csv(DATA_PATH, header=None, names=columns)
display(df.head())

,Sex,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight,Rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


## 4. Làm sạch cơ bản

Theo kết quả EDA, dữ liệu không có missing value. Ở đây ta kiểm tra lại nhanh và loại bỏ duplicate nếu có để đảm bảo dữ liệu sạch trước khi chia train/test.

In [15]:
missing_count = int(df.isna().sum().sum())
duplicate_count_before = int(df.duplicated().sum())

df_clean = df.drop_duplicates().copy()
duplicate_count_after = int(df_clean.duplicated().sum())

print(f'Tổng số giá trị thiếu: {missing_count}')
print(f'Số dòng trùng lặp trước khi làm sạch: {duplicate_count_before}')
print(f'Số dòng trùng lặp sau khi làm sạch: {duplicate_count_after}')
print(f'Kích thước dữ liệu sau làm sạch: {df_clean.shape}')

Tổng số giá trị thiếu: 0
Số dòng trùng lặp trước khi làm sạch: 0
Số dòng trùng lặp sau khi làm sạch: 0
Kích thước dữ liệu sau làm sạch: (4177, 9)


## 5. Xác định nhóm đặc trưng

In [16]:
target_col = 'Rings'
categorical_cols = ['Sex']
size_cols = ['Length', 'Diameter', 'Height']
weight_cols = ['WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']
numeric_cols = size_cols + weight_cols

x = df_clean.drop(columns=[target_col]) # Tách biến độc lập
y = df_clean[target_col] # Tách biến mục tiêu

print('Biến phân loại:', categorical_cols)
print('Biến kích thước:', size_cols)
print('Biến khối lượng:', weight_cols)
print('Biến mục tiêu:', target_col)

Biến phân loại: ['Sex']
Biến kích thước: ['Length', 'Diameter', 'Height']
Biến khối lượng: ['WholeWeight', 'ShuckedWeight', 'VisceraWeight', 'ShellWeight']
Biến mục tiêu: Rings


## 6. Chia train/test 

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.3,
    random_state=42,
)

print(f'X_train: {X_train.shape}')
print(f'X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')

X_train: (2923, 8)
X_test: (1254, 8)
y_train: (2923,)
y_test: (1254,)


## 7. Hàm hỗ trợ biến đổi dữ liệu

Để tránh rò rỉ dữ liệu, mọi encoder và scaler đều chỉ được `fit` trên tập train rồi mới `transform` sang tập test.

In [18]:
# Giải quyết sự thiếu nhất quán thư viện scikit-learn về tham số sparse/sparse_output của OneHotEncoder giữa các phiên bản
# handle_unknown='ignore' để xử lý các giá trị không xuất hiện trong tập huấn luyện, sparse_output=False hoặc sparse=False để trả về mảng numpy thay vì ma trận thưa.
def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def transform_to_dataframe(preprocessor, X_train_df, X_test_df):
    X_train_transformed = preprocessor.fit_transform(X_train_df)
    # chỉ biến đổi Xtest dựa trên tham số đã học được từ Xtrain để tránh rò rỉ dữ liệu.
    X_test_transformed = preprocessor.transform(X_test_df)
    # get_feature_names_out() để lấy tên các tính năng sau khi biến đổi, đặc biệt hữu ích khi sử dụng OneHotEncoder để biết được tên các cột mới được tạo ra.
    feature_names = preprocessor.get_feature_names_out()

    X_train_result = pd.DataFrame(X_train_transformed, columns=feature_names, index=X_train_df.index)
    X_test_result = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test_df.index)
    return X_train_result, X_test_result

## 8. Phiên bản 1: `encoded_only`

Phiên bản này chỉ mã hóa `Sex`, còn các biến số được giữ nguyên. Đây là phiên bản phù hợp để dùng làm baseline chung và đặc biệt phù hợp với các mô hình cây như Decision Tree, Random Forest, Gradient Boosting.

In [19]:
encoded_preprocessor = ColumnTransformer(
    transformers=[
        ('sex', make_onehot_encoder(), categorical_cols),
    ],
    # remainder = 'passthrough' để giữ nguyên các cột không được liệt kê trong categorial cols, nghĩa là các cột kích thước và khối lượng sẽ được giữ nguyên mà không bị biến đổi.
    remainder='passthrough',
    # verbose_feature_names_out=False để giữ nguyên tên cột gốc cho các cột không được biến đổi, thay vì thêm tiền tố transformer__column_name.
    verbose_feature_names_out=False,
)

# Biến đổi dữ liệu và chuyển đổi kết quả thành DataFrame để dễ dàng thao tác và phân tích sau này.
X_train_encoded, X_test_encoded = transform_to_dataframe(encoded_preprocessor, X_train, X_test)
display(X_train_encoded.head())
print(X_train_encoded.shape, X_test_encoded.shape)

,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
2830,1.0,0.0,0.0,0.525,0.430,0.135,0.8435,0.4325,0.1800,0.1815
925,0.0,1.0,0.0,0.430,0.325,0.100,0.3645,0.1575,0.0825,0.1050
3845,0.0,0.0,1.0,0.455,0.350,0.105,0.4160,0.1625,0.0970,0.1450
547,0.0,0.0,1.0,0.205,0.155,0.045,0.0425,0.0170,0.0055,0.0155
2259,1.0,0.0,0.0,0.590,0.465,0.160,1.1005,0.5060,0.2525,0.2950


(2923, 10) (1254, 10)


## 9. Phiên bản 2: `standard_scaled`

Phiên bản này mã hóa `Sex` và chuẩn hóa toàn bộ biến số bằng `StandardScaler`. Nó phù hợp hơn cho các mô hình nhạy với scale như Linear Regression, Ridge, KNN, SVR, LinearSVR, SGDRegressor và MLPRegressor.

In [20]:
# Tạo một pipeline tiền xử lý chuẩn hóa dữ liệu, trong đó biến phân loại được one-hot encoding và biến số được chuẩn hóa bằng StandardScaler.
# standardScaler là một kỹ thuật chuẩn hóa dữ liệu phổ biến, giúp đưa các biến số về cùng một thang đo với trung bình bằng 0 và độ lệch chuẩn bằng 1
# theo công thức: z = (x - mean) / std, trong đó mean là trung bình của biến số và std là độ lệch chuẩn của biến số. Điều này giúp cải thiện hiệu suất của nhiều thuật toán học máy bằng cách đảm bảo rằng tất cả các biến số đều có cùng một thang đo.
standard_preprocessor = ColumnTransformer(
    transformers=[
        ('sex', make_onehot_encoder(), categorical_cols),
        ('num', StandardScaler(), numeric_cols),
    ],
    # drop là một tùy chọn để loại bỏ các cột không được liệt kê trong transformers, nghĩa là nếu có bất kỳ cột nào không được biến đổi bởi OneHotEncoder hoặc StandardScaler, chúng sẽ bị loại bỏ khỏi kết quả cuối cùng.
    remainder='drop',
    verbose_feature_names_out=False,
)

X_train_standard, X_test_standard = transform_to_dataframe(standard_preprocessor, X_train, X_test)
display(X_train_standard.head())
print(X_train_standard.shape, X_test_standard.shape)

,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
2830,1.0,0.0,0.0,-0.009546,0.206802,-0.120724,0.014578,0.311054,-0.020982,-0.425408
925,0.0,1.0,0.0,-0.803180,-0.854049,-0.944651,-0.959009,-0.919416,-0.913759,-0.969745
3845,0.0,0.0,1.0,-0.594329,-0.601465,-0.826947,-0.854333,-0.897043,-0.780987,-0.685124
547,0.0,0.0,1.0,-2.682841,-2.571616,-2.239393,-1.613488,-1.548074,-1.618824,-1.606585
2259,1.0,0.0,0.0,0.533467,0.560419,0.467795,0.536941,0.639926,0.642877,0.382204


(2923, 10) (1254, 10)


## 10. Phiên bản 3: `robust_log_scaled`

Đây là phương án tiền xử lý bổ sung dựa trên kết quả EDA: các biến khối lượng có độ lệch và ngoại lệ, nên ta thử `log1p` cho nhóm khối lượng rồi chuẩn hóa bằng `RobustScaler` để giảm ảnh hưởng của outliers.

In [21]:
X_train_robust_input = X_train.copy()
X_test_robust_input = X_test.copy()

X_train_robust_input[weight_cols] = np.log1p(X_train_robust_input[weight_cols])
X_test_robust_input[weight_cols] = np.log1p(X_test_robust_input[weight_cols])

In [22]:
display(X_train_robust_input[weight_cols].head())

,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
2830,0.611666,0.359421,0.165514,0.166785
925,0.310788,0.146263,0.079273,0.099845
3845,0.347836,0.150573,0.092579,0.135405
547,0.041622,0.016857,0.005485,0.015381
2259,0.742175,0.409457,0.225142,0.258511


In [23]:
robust_preprocessor = ColumnTransformer(
    transformers=[
        ('sex', make_onehot_encoder(), categorical_cols),
        ('num', RobustScaler(), numeric_cols),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

X_train_robust, X_test_robust = transform_to_dataframe(robust_preprocessor, X_train_robust_input, X_test_robust_input)
display(X_train_robust.head())
print(X_train_robust.shape, X_test_robust.shape)

,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
2830,1.0,0.0,0.0,-0.121212,0.037037,-0.2,0.047566,0.278688,0.043230,-0.271759
925,0.0,1.0,0.0,-0.696970,-0.740741,-0.9,-0.699066,-0.611250,-0.583367,-0.682528
3845,0.0,0.0,1.0,-0.545455,-0.555556,-0.8,-0.607131,-0.593254,-0.486691,-0.464321
547,0.0,0.0,1.0,-2.060606,-2.000000,-2.0,-1.367006,-1.151517,-1.119485,-1.200836
2259,1.0,0.0,0.0,0.272727,0.296296,0.3,0.371427,0.487588,0.476458,0.291110


(2923, 10) (1254, 10)


## 11. Ghép lại với biến mục tiêu và lưu dữ liệu

Ta lưu cả tập raw đã chia và các phiên bản đã tiền xử lý để notebook huấn luyện mô hình có thể dùng lại trực tiếp.

In [24]:
interim_dir = PROJECT_ROOT / 'data' / 'interim'
processed_dir = PROJECT_ROOT / 'data' / 'processed'
interim_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)


def attach_target(X_df, y_series, target_name='Rings'):
    result = X_df.copy()
    result[target_name] = y_series.loc[X_df.index]
    return result


train_raw_df = attach_target(X_train, y_train)
test_raw_df = attach_target(X_test, y_test)
train_encoded_df = attach_target(X_train_encoded, y_train)
test_encoded_df = attach_target(X_test_encoded, y_test)
train_standard_df = attach_target(X_train_standard, y_train)
test_standard_df = attach_target(X_test_standard, y_test)
train_robust_df = attach_target(X_train_robust, y_train)
test_robust_df = attach_target(X_test_robust, y_test)

train_raw_df.to_csv(interim_dir / 'abalone_train_raw.csv', index=False)
test_raw_df.to_csv(interim_dir / 'abalone_test_raw.csv', index=False)
train_encoded_df.to_csv(processed_dir / 'abalone_train_encoded.csv', index=False)
test_encoded_df.to_csv(processed_dir / 'abalone_test_encoded.csv', index=False)
train_standard_df.to_csv(processed_dir / 'abalone_train_standard.csv', index=False)
test_standard_df.to_csv(processed_dir / 'abalone_test_standard.csv', index=False)
train_robust_df.to_csv(processed_dir / 'abalone_train_robust_log.csv', index=False)
test_robust_df.to_csv(processed_dir / 'abalone_test_robust_log.csv', index=False)

preprocessing_metadata = {
    'dataset': 'Abalone Age',
    'target': target_col,
    'train_ratio': 0.7,
    'test_ratio': 0.3,
    'random_state': 42,
    'categorical_cols': categorical_cols,
    'numeric_cols': numeric_cols,
    'size_cols': size_cols,
    'weight_cols': weight_cols,
    'artifacts': {
        'train_raw': 'data/interim/abalone_train_raw.csv',
        'test_raw': 'data/interim/abalone_test_raw.csv',
        'train_encoded': 'data/processed/abalone_train_encoded.csv',
        'test_encoded': 'data/processed/abalone_test_encoded.csv',
        'train_standard': 'data/processed/abalone_train_standard.csv',
        'test_standard': 'data/processed/abalone_test_standard.csv',
        'train_robust_log': 'data/processed/abalone_train_robust_log.csv',
        'test_robust_log': 'data/processed/abalone_test_robust_log.csv',
    },
}

with open(processed_dir / 'preprocessing_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(preprocessing_metadata, f, ensure_ascii=False, indent=2)

display(pd.DataFrame({'saved_file': list(preprocessing_metadata['artifacts'].values())}))

,saved_file
0,data/interim/abalone_train_raw.csv
1,data/interim/abalone_test_raw.csv
2,data/processed/abalone_train_encoded.csv
3,data/processed/abalone_test_encoded.csv
4,data/processed/abalone_train_standard.csv
5,data/processed/abalone_test_standard.csv
6,data/processed/abalone_train_robust_log.csv
7,data/processed/abalone_test_robust_log.csv


## 12. Kết luận bước tiền xử lý

- Dữ liệu đã được làm sạch cơ bản và chia đúng theo tỷ lệ `70/30` với `random_state = 42`.
- Biến `Sex` đã được mã hóa để các mô hình scikit-learn có thể sử dụng.
- Đã chuẩn bị ba phiên bản dữ liệu cho ba mục đích khác nhau:
  - `encoded_only`: phù hợp cho các mô hình cây và làm baseline chung.
  - `standard_scaled`: phù hợp cho các mô hình nhạy với scale.
  - `robust_log_scaled`: là phương án cải thiện để kiểm tra tác động của outlier và skewness.
- Các file đầu ra đã được lưu trong `data/interim` và `data/processed`, sẵn sàng cho bước huấn luyện mô hình.